# 00 - Environment check

Step 0.1 of `THESIS_GUIDE.md`. Verifies, top to bottom, that the environment can run the rest of the thesis:

1. the notebook runs on the project `.venv` interpreter;
2. every required library imports (including scipy, previously blocked by Windows Application Control);
3. PyTorch sees the GPU and can compute on it;
4. the `xaifin` package is installed in editable mode (points to `code/xaifin/`);
5. the core datasets for the thesis universes are present;
6. `captum`, `shap` and `lime` produce correct attributions on a tiny MLP with the same input shape as the real models (`[N, T, F] -> [N]`);
7. parquet I/O works (results store).

Every check records a pass/fail. The last cell raises if anything failed, so **"runs top to bottom with no errors" means the environment is ready**.

In [1]:
import importlib
import importlib.metadata as md
import platform
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

CHECKS: dict[str, bool] = {}


def record(name: str, ok: bool, detail: str = "") -> None:
    CHECKS[name] = bool(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}{' - ' + detail if detail else ''}")

## 1. Interpreter

The kernel must be the project's `.venv` (Python 3.12), otherwise the packages installed there are not visible.

In [2]:
print("Python    :", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform  :", platform.platform())

record("interpreter is project .venv", ".venv" in Path(sys.executable).parts, sys.prefix)
record("python >= 3.10", sys.version_info >= (3, 10))

Python    : 3.12.10
Executable: D:\CodeProjects\master-thesis\.venv\Scripts\python.exe
Platform  : Windows-11-10.0.26200-SP0
[PASS] interpreter is project .venv - D:\CodeProjects\master-thesis\.venv
[PASS] python >= 3.10


## 2. Libraries

Import every library the thesis depends on and record its version. `import_name` differs from the distribution name for a few packages (e.g. `scikit-learn` -> `sklearn`).

In [3]:
LIBRARIES = {
    # distribution    : import name
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy.stats",  # scipy.stats pulls in the compiled modules that were blocked
    "scikit-learn": "sklearn.linear_model",
    "statsmodels": "statsmodels.api",
    "torch": "torch",
    "lightgbm": "lightgbm",
    "xgboost": "xgboost",
    "shap": "shap",
    "captum": "captum.attr",
    "lime": "lime.lime_tabular",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "streamlit": "streamlit",
    "pyarrow": "pyarrow",
    "xaifin": "xaifin",
}

rows = []
for dist, module in LIBRARIES.items():
    try:
        importlib.import_module(module)
        rows.append({"package": dist, "version": md.version(dist), "status": "ok", "error": ""})
    except Exception as e:  # ImportError, or OSError for blocked DLLs
        rows.append({"package": dist, "version": "-", "status": "FAIL", "error": f"{type(e).__name__}: {e}"[:120]})

versions = pd.DataFrame(rows).set_index("package")
display(versions)

failed = versions.index[versions["status"] != "ok"].tolist()
record("all libraries import", not failed, f"failed: {failed}" if failed else f"{len(versions)} packages")

,version,status,error
package,,,
numpy,2.5.2,ok,
pandas,2.3.3,ok,
scipy,1.18.1,ok,
scikit-learn,1.9.0,ok,
statsmodels,0.15.0,ok,
torch,2.11.0+cu128,ok,
lightgbm,4.7.0,ok,
xgboost,2.1.4,ok,
shap,0.52.0,ok,


[PASS] all libraries import - 17 packages


## 3. PyTorch and CUDA

The CUDA build of torch must come from the PyTorch index (see `requirements.txt`). A CPU-only build would show a version without the `+cu...` suffix.

In [4]:
import torch

print("torch      :", torch.__version__)
print("CUDA build :", torch.version.cuda)

cuda_ok = torch.cuda.is_available()
record("CUDA available", cuda_ok)

if cuda_ok:
    print("Device     :", torch.cuda.get_device_name(0))
    print("Capability :", torch.cuda.get_device_capability(0))
    print(f"Memory     : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    a = torch.randn(1024, 1024, device="cuda")
    gpu = (a @ a.T).cpu()
    cpu = a.cpu() @ a.cpu().T
    record("GPU matmul matches CPU", torch.allclose(gpu, cpu, rtol=1e-3, atol=1e-2))

DEVICE = torch.device("cuda" if cuda_ok else "cpu")

torch      : 2.11.0+cu128
CUDA build : 12.8
[PASS] CUDA available
Device     : NVIDIA GeForce RTX 5090
Capability : (12, 0)
Memory     : 31.8 GB
[PASS] GPU matmul matches CPU


## 4. `xaifin` editable install

`pip install -e code` links the package to the source folder, so `xaifin.__file__` must point into `code/xaifin/` rather than into `site-packages`. All later paths are derived from it, so notebooks don't depend on the current working directory.

In [5]:
import xaifin

CODE_ROOT = Path(xaifin.__file__).resolve().parents[1]  # .../master-thesis/code
DATA_ROOT = CODE_ROOT / "data"
RESULTS_ROOT = CODE_ROOT / "results"

print("xaifin     :", xaifin.__version__, "->", xaifin.__file__)
print("CODE_ROOT  :", CODE_ROOT)

record("xaifin is an editable install", "site-packages" not in Path(xaifin.__file__).parts)
record("code/ layout found", (CODE_ROOT / "pyproject.toml").exists() and DATA_ROOT.exists())

xaifin     : 0.1.0 -> D:\CodeProjects\master-thesis\code\xaifin\__init__.py
CODE_ROOT  : D:\CodeProjects\master-thesis\code
[PASS] xaifin is an editable install
[PASS] code/ layout found


## 5. Data availability

The thesis universes (DJI, NASDAQ100, SX5E) need: OHLCV (for labels and portfolios), Alpha158 and Alpha360 features, market gate features (MASTER/MATCC), and the inc-matrix and market-cap files (HIST/DiscoverPLF). Plus index constituents.

In [6]:
UNIVERSES = {"dji": "us", "nasdaq100": "us", "sx5e": "eu"}


def required_files(universe: str, nation: str) -> list[Path]:
    d = DATA_ROOT / universe
    return [
        d / f"{universe}.csv",
        d / f"{universe}_alpha158.csv",
        d / f"{universe}_alpha360.csv",
        d / f"{nation}_market.csv",
        d / f"{universe}_inc_matrix.npz",
        d / f"{universe}_market_cap.csv",
        DATA_ROOT / "constituents" / "eodhd" / f"{universe}.csv",
    ]


rows = []
for universe, nation in UNIVERSES.items():
    for path in required_files(universe, nation):
        rows.append({
            "universe": universe,
            "file": path.relative_to(DATA_ROOT).as_posix(),
            "exists": path.exists(),
            "size_MB": round(path.stat().st_size / 1024**2, 1) if path.exists() else None,
        })

data_files = pd.DataFrame(rows)
display(data_files)

missing = data_files.loc[~data_files["exists"], "file"].tolist()
record("all thesis data files present", not missing, f"missing: {missing}" if missing else "")

,universe,file,exists,size_MB
0,dji,dji/dji.csv,True,34.6
1,dji,dji/dji_alpha158.csv,True,713.5
2,dji,dji/dji_alpha360.csv,True,1751.7
3,dji,dji/us_market.csv,True,8.5
4,dji,dji/dji_inc_matrix.npz,True,0.0
5,dji,dji/dji_market_cap.csv,True,62.4
6,dji,constituents/eodhd/dji.csv,True,0.0
7,nasdaq100,nasdaq100/nasdaq100.csv,True,114.7
8,nasdaq100,nasdaq100/nasdaq100_alpha158.csv,True,2675.2
9,nasdaq100,nasdaq100/nasdaq100_alpha360.csv,True,6579.5


[PASS] all thesis data files present


Quick look at the feature files' headers, without loading the full CSVs.

- **Alpha158 has 157 features here**, not 158: FinBench computes it without a VWAP input, so Qlib's `VWAP0` is missing (9 K-bar + `OPEN0`/`HIGH0`/`LOW0` + 29 rolling operators x 5 windows). This is why FinBench uses `gate_input_start_index=157` (MASTER/MATCC) and `num_latent=157` (FactorVAE).
- **Alpha360** has 6 series (CLOSE, OPEN, HIGH, LOW, VOLUME, VWAP) x 60 lags = 360.

In [7]:
for universe in UNIVERSES:
    for kind, expected in [("alpha158", 157), ("alpha360", 360)]:
        path = DATA_ROOT / universe / f"{universe}_{kind}.csv"
        if not path.exists():
            continue
        head = pd.read_csv(path, nrows=5)
        n_features = head.shape[1] - 2  # minus 'instrument' and 'date'
        record(f"{universe} {kind} has {expected} features", n_features == expected, f"found {n_features}")

[PASS] dji alpha158 has 157 features - found 157
[PASS] dji alpha360 has 360 features - found 360
[PASS] nasdaq100 alpha158 has 157 features - found 157
[PASS] nasdaq100 alpha360 has 360 features - found 360
[PASS] sx5e alpha158 has 157 features - found 157
[PASS] sx5e alpha360 has 360 features - found 360


## 6. XAI smoke test on a tiny MLP

A toy model with the **same interface as the real adapters** (`THESIS_GUIDE.md` §2.3): input `[N, T, F]`, output `[N]`. Baseline = zeros, which after the robust z-score corresponds to the training median.

Each library is checked against a property the method guarantees, not only that it runs:
- **Integrated Gradients** (captum): *completeness*, sum of attributions = f(x) - f(baseline);
- **KernelSHAP** (shap): *additivity*, expected value + sum of SHAP values = f(x);
- **LIME**: returns a local linear surrogate whose prediction is close to f(x).

In [8]:
torch.manual_seed(42)
np.random.seed(42)

N, T, F = 8, 5, 4  # stocks, lookback days, features


class TinyMLP(torch.nn.Module):
    """[N, T, F] -> [N], the same contract as a xaifin ModelAdapter.forward."""

    def __init__(self, T: int, F: int, hidden: int = 16):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(), torch.nn.Linear(T * F, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


model = TinyMLP(T, F).to(DEVICE).eval()
x = torch.randn(N, T, F, device=DEVICE)
baseline = torch.zeros_like(x)

with torch.no_grad():
    y = model(x)
print("input :", tuple(x.shape), "-> output:", tuple(y.shape))
record("tiny model forward [N,T,F] -> [N]", y.shape == (N,))

input : (8, 5, 4) -> output: (8,)
[PASS] tiny model forward [N,T,F] -> [N]


In [9]:
from captum.attr import IntegratedGradients

ig = IntegratedGradients(model)
attr, delta = ig.attribute(x, baselines=baseline, n_steps=200, return_convergence_delta=True)

print("IG attributions:", tuple(attr.shape), "(one [T, F] map per stock)")
print("max |completeness error|:", delta.abs().max().item())

record("captum IG shape [N,T,F]", attr.shape == x.shape)
record("captum IG completeness", delta.abs().max().item() < 1e-2)

IG attributions: (8, 5, 4) (one [T, F] map per stock)
max |completeness error|: 5.960464477539063e-08
[PASS] captum IG shape [N,T,F]
[PASS] captum IG completeness


In [10]:
import shap


def predict_flat(X: np.ndarray) -> np.ndarray:
    """numpy [n, T*F] -> numpy [n]: the black-box view used by KernelSHAP and LIME."""
    with torch.no_grad():
        xt = torch.as_tensor(X, dtype=torch.float32, device=DEVICE).reshape(-1, T, F)
        return model(xt).cpu().numpy()


x0 = x[:1].reshape(1, -1).cpu().numpy()  # one stock, flattened to T*F = 20 players
explainer = shap.KernelExplainer(predict_flat, np.zeros((1, T * F)))
phi = explainer.shap_values(x0, nsamples=500, silent=True)

reconstructed = explainer.expected_value + phi.sum()
fx = predict_flat(x0)[0]
print(f"f(x) = {fx:.5f} | E[f] + sum(SHAP) = {float(reconstructed):.5f}")

record("shap KernelSHAP shape", np.asarray(phi).reshape(-1).shape == (T * F,))
record("shap KernelSHAP additivity", abs(float(reconstructed) - fx) < 1e-4)

f(x) = 0.44557 | E[f] + sum(SHAP) = 0.44557
[PASS] shap KernelSHAP shape
[PASS] shap KernelSHAP additivity


In [11]:
from lime.lime_tabular import LimeTabularExplainer

feature_names = [f"f{f}_t-{T - 1 - t}" for t in range(T) for f in range(F)]  # matches flatten order
background = np.random.randn(500, T * F)

lime_explainer = LimeTabularExplainer(
    background, mode="regression", feature_names=feature_names, discretize_continuous=False, random_state=42
)
exp = lime_explainer.explain_instance(x0[0], predict_flat, num_features=T * F, num_samples=2000)

print("top 5 LIME weights:", [(name, round(w, 4)) for name, w in exp.as_list()[:5]])
print(f"f(x) = {fx:.5f} | LIME local prediction = {exp.local_pred[0]:.5f} | R^2 = {exp.score:.3f}")

record("lime explanation for all features", len(exp.as_list()) == T * F)
record("lime surrogate close to f(x)", abs(exp.local_pred[0] - fx) < 0.1)

top 5 LIME weights: [('f3_t-1', 0.1245), ('f0_t-4', -0.0984), ('f3_t-4', 0.0978), ('f2_t-2', 0.0874), ('f3_t-3', -0.0853)]
f(x) = 0.44557 | LIME local prediction = 0.47816 | R^2 = 0.980
[PASS] lime explanation for all features
[PASS] lime surrogate close to f(x)


Sanity cross-check: IG and KernelSHAP are different methods, but on a small smooth model with the same zero baseline they should broadly agree on feature ranking.

In [12]:
ig_stock0 = attr[0].reshape(-1).detach().cpu().numpy()
shap_stock0 = np.asarray(phi).reshape(-1)

rank_corr = pd.Series(ig_stock0).rank().corr(pd.Series(shap_stock0).rank())
print(f"Spearman(IG, KernelSHAP) on stock 0: {rank_corr:.3f}")
record("IG and KernelSHAP agree (Spearman > 0.8)", rank_corr > 0.8)

Spearman(IG, KernelSHAP) on stock 0: 0.936
[PASS] IG and KernelSHAP agree (Spearman > 0.8)


## 7. Parquet I/O (results store)

In [13]:
frame = pd.DataFrame({"feature": feature_names, "ig": ig_stock0, "shap": shap_stock0})

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "global.parquet"
    frame.to_parquet(path, index=False)
    round_trip = pd.read_parquet(path)

record("parquet round-trip", round_trip.equals(frame))

[PASS] parquet round-trip


## Summary

In [14]:
summary = pd.Series(CHECKS, name="passed").to_frame()
display(summary)

failed = summary.index[~summary["passed"]].tolist()
if failed:
    raise RuntimeError(f"{len(failed)} environment check(s) failed: {failed}")
print(f"All {len(summary)} checks passed - environment ready.")

,passed
interpreter is project .venv,True
python >= 3.10,True
all libraries import,True
CUDA available,True
GPU matmul matches CPU,True
xaifin is an editable install,True
code/ layout found,True
all thesis data files present,True
dji alpha158 has 157 features,True
dji alpha360 has 360 features,True


All 23 checks passed - environment ready.
